# Bloomberg P&L Data Extraction

## STAGE 2 — Data Cleaning & Restructuring using Python

### 1. Read the extract with Pandas

In [568]:
#import libraries
import pandas as pd
import numpy as np 
import openpyxl as pxl

In [569]:
raw = pd.read_csv(
    r"C:\Users\HP\OneDrive\Desktop\innovexis internship\project8\ADS_GY_Equity_RAW_EXPORT.csv",
    header=None
)


I loaded the raw CSV file into Python without using the first row as headers because the file contains metadata before the actual financial table.

### Why is starting with header=None (or an explicit skiprows) safer than letting pandas guess?
I used header=None when initially reading the extract because the Bloomberg-style worksheet contains metadata, blank rows and a two-row period header before the actual P&L table. Allowing Pandas to guess the header could cause the wrong row to become the column names and could shift or misinterpret the financial data. Reading the sheet without assuming a header allows the structure to be inspected before defining the actual table.


In [570]:
raw
# I displayed the raw data to understand its structure and identify where the actual P&L table starts.



,0,1,2,3,4,5,6,7,8,9
0,Bloomberg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ADS GY Equity - ADIDAS AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Income Statement | Standardized | Peri...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Currency: EUR Units: Millions (except p...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,*** SYNTHETIC SAMPLE DATA - GENERATED FOR CLAS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,In Millions of EUR except Per Share,NaN,FY 2016,FY 2017,FY 2018,FY 2019,FY 2020,FY 2021,FY 2022,FY 2023
7,NaN,NaN,12/31/2016,12/31/2017,12/31/2018,12/31/2019,12/31/2020,12/31/2021,12/31/2022,12/31/2023
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Revenue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [571]:
raw = raw.iloc[9:]
# I removed the first nine rows because they contained metadata and not financial data.
  

In [572]:
print("Raw shape:", raw.shape)
print(raw.iloc[0:15, [0, 2, 3, 4]].to_string(index=True))

Raw shape: (48, 10)
                                  0          2           3           4
9                           Revenue        NaN         NaN         NaN
10                        Wholesale   11,663.0    12,943.0    12,733.0
11                          Retail    5,841.0     7,172.0     8,196.0 
12                Other  Businesses      979.0     1,103.0     986,000
13                Total Net Revenue   18,483.0    21,218.0    21,915.0
14              Revenue Growth  Y/Y         NM       14.8%        3.3%
15                Total Net Revenue   18,483.0    21,218.0    21,915.0
16                              NaN        NaN         NaN         NaN
17               Cost of Goods Sold  (9,500.0)  (10,524.0)  (10,607.0)
18                     GROSS PROFIT    8,983.0    10,694.0    11,308.0
19                   Gross Margin %      48.6%       50.4%       51.6%
20                              NaN        NaN         NaN         NaN
21    Royalty and Commission Income      108.0       115.

I checked the size of the dataset and displayed selected rows and columns to confirm that the financial data was in the expected position.



In [573]:
print(raw.iloc[38:48, [0, 2, 3, 4]].to_string(index=True))

                                                                                                                                             0      2      3      4
47                                                                                                              Wtd. Avg Shares - Diluted (mn)  201.8  203.3  200.8
48                                                                                                                    Dividend Per Share (DPS)   2.00   2.60   3.35
49                                                                                                                                         NaN    NaN    NaN    NaN
50                                                                                                                                         NaN    NaN    NaN    NaN
51                                              Source: Bloomberg Standardized Financials  ---  SYNTHETIC SAMPLE, generated for classroom use.    NaN    NaN    NaN
52  (1) FY2021 r

In [574]:
print(raw.head(20).to_string())

                                           0    1          2           3           4           5          6           7           8           9
9                                    Revenue  NaN        NaN         NaN         NaN         NaN        NaN         NaN         NaN         NaN
10                                 Wholesale  NaN   11,663.0    12,943.0    12,733.0    13,191.0   10,160.0    11,148.0    12,156.0    11,142.0
11                                   Retail   NaN   5,841.0     7,172.0     8,196.0     9,480.0    8,970.0     9,428.0     9,725.0     9,706.0 
12                         Other  Businesses  (2)      979.0     1,103.0     986,000       969.0      714.0       658.0       630.0       579.0
13                         Total Net Revenue  NaN   18,483.0    21,218.0    21,915.0    23,640.0   19,844.0    21,234.0    22,511.0    21,427.0
14                       Revenue Growth  Y/Y  NaN         NM       14.8%        3.3%        7.9%     -16.1%        7.0%        6.0%     

In [575]:
# P&L table only
pnl = raw.iloc[0:40].copy()

pnl = pnl.reset_index(drop=True)

print(pnl)



                                           0    1          2           3  \
0                                    Revenue  NaN        NaN         NaN   
1                                  Wholesale  NaN   11,663.0    12,943.0   
2                                    Retail   NaN   5,841.0     7,172.0    
3                          Other  Businesses  (2)      979.0     1,103.0   
4                          Total Net Revenue  NaN   18,483.0    21,218.0   
5                        Revenue Growth  Y/Y  NaN         NM       14.8%   
6                          Total Net Revenue  NaN   18,483.0    21,218.0   
7                                        NaN  NaN        NaN         NaN   
8                         Cost of Goods Sold  NaN  (9,500.0)  (10,524.0)   
9                               GROSS PROFIT  NaN    8,983.0    10,694.0   
10                            Gross Margin %  NaN      48.6%       50.4%   
11                                       NaN  NaN        NaN         NaN   
12          

I selected the rows containing the P&L data and reset the index to create a clean and easier-to-work-with table.

In [576]:
pnl.head(20)

,0,1,2,3,4,5,6,7,8,9
0,Revenue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Wholesale,NaN,"11,663.0","12,943.0","12,733.0","13,191.0","10,160.0","11,148.0","12,156.0","11,142.0"
2,Retail,NaN,"5,841.0","7,172.0","8,196.0","9,480.0","8,970.0","9,428.0","9,725.0","9,706.0"
3,Other Businesses,(2),979.0,"1,103.0","986,000",969.0,714.0,658.0,630.0,579.0
4,Total Net Revenue,NaN,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"
5,Revenue Growth Y/Y,NaN,NM,14.8%,3.3%,7.9%,-16.1%,7.0%,6.0%,-4.8%
6,Total Net Revenue,NaN,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Cost of Goods Sold,NaN,"(9,500.0)","(10,524.0)","(10,607.0)","(11,347.0)","(9,982.0)","(10,511.0)","(11,863.0)","(11,249.0)"
9,GROSS PROFIT,NaN,"8,983.0","10,694.0","11,308.0","12,293.0","9,862.0","10,723.0","10,648.0","10,178.0"


In [577]:
print(pnl.iloc[38:48, [0, 2, 3, 4]].to_string(index=True))

                                   0      2      3      4
38    Wtd. Avg Shares - Diluted (mn)  201.8  203.3  200.8
39          Dividend Per Share (DPS)   2.00   2.60   3.35


### 2. Create a function to convert text to numbers


In [578]:
def clean_numeric(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Handle placeholders
    if value in ["", "NM", "#N/A N/A", "N/A", "NA", "—", "â€”"]:
        return np.nan

    # Convert parentheses to negative numbers
    if value.startswith("(") and value.endswith(")"):
        value = "-" + value[1:-1]

    # Handle percentages
    if value.endswith("%"):
        value = value[:-1].strip()
        return float(value) / 100

    # Remove thousands separators
    value = value.replace(",", "")

    # Remove unnecessary spaces
    value = value.replace(" ", "")

    try:
        return float(value)
    except ValueError:
        return np.nan


In [579]:
examples = [
    "11,663.00",
    "-16.10%",
    "(500.00)"
]

for x in examples:
    print(x, "->", clean_numeric(x))


11,663.00 -> 11663.0
-16.10% -> -0.161
(500.00) -> -500.0


I created a function to convert the raw text values into numeric values.

The function handles:

* commas used as thousands separators
* leading and trailing spaces
* parentheses used for negative numbers
* percentage signs
* placeholders such as — and #N/A N/A NM values

### 3. Deal with duplicated rows

I identified duplicated rows by comparing the line-item name and the values across all fiscal years.

The extract contains Total Net Revenue twice. The two rows contain the same values for FY2016–FY2023, so the second occurrence is a duplicate presentation row rather than a separate financial item.

In [580]:
duplicate_rows = pnl.duplicated(keep=False)

pnl[duplicate_rows]

# I checked for duplicate rows

,0,1,2,3,4,5,6,7,8,9
4,Total Net Revenue,NaN,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"
6,Total Net Revenue,NaN,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [581]:
# Remove exact duplicate records while keeping the first occurrence.
before = len(pnl)
pnl = pnl.drop_duplicates(keep="first").reset_index(drop=True)

print(f"Removed {before - len(pnl)} duplicate rows")
pnl[pnl.iloc[:, 0].astype(str).str.strip() == "Total Net Revenue"]

Removed 4 duplicate rows


,0,1,2,3,4,5,6,7,8,9
4,Total Net Revenue,NaN,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"


In [582]:
pnl.head(20)

,0,1,2,3,4,5,6,7,8,9
0,Revenue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Wholesale,NaN,"11,663.0","12,943.0","12,733.0","13,191.0","10,160.0","11,148.0","12,156.0","11,142.0"
2,Retail,NaN,"5,841.0","7,172.0","8,196.0","9,480.0","8,970.0","9,428.0","9,725.0","9,706.0"
3,Other Businesses,(2),979.0,"1,103.0","986,000",969.0,714.0,658.0,630.0,579.0
4,Total Net Revenue,NaN,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"
5,Revenue Growth Y/Y,NaN,NM,14.8%,3.3%,7.9%,-16.1%,7.0%,6.0%,-4.8%
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Cost of Goods Sold,NaN,"(9,500.0)","(10,524.0)","(10,607.0)","(11,347.0)","(9,982.0)","(10,511.0)","(11,863.0)","(11,249.0)"
8,GROSS PROFIT,NaN,"8,983.0","10,694.0","11,308.0","12,293.0","9,862.0","10,723.0","10,648.0","10,178.0"
9,Gross Margin %,NaN,48.6%,50.4%,51.6%,52.0%,49.7%,50.5%,47.3%,47.5%


### 4. Normalise the labels

I standardized the line-item labels by removing unnecessary whitespace and applying consistent capitalization.

In [583]:
pnl[0] = (
    pnl[0]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

#I removed unnecessary spaces from the line-item names to make matching and lookups more reliable.

In [584]:
pnl[0] = pnl[0].str.lower()
# I converted all line-item names to lowercase so that the labels had a consistent format.


## 5. Reshape wide data into tidy/long format

In [585]:
pnl.columns = [
    "Line Item",
    "Footnote",
    "FY2016",
    "FY2017",
    "FY2018",
    "FY2019",
    "FY2020",
    "FY2021",
    "FY2022",
    "FY2023"
]
# I assigned clear column names to identify the line items, footnotes and fiscal years.

In [586]:
pnl

,Line Item,Footnote,FY2016,FY2017,FY2018,FY2019,FY2020,FY2021,FY2022,FY2023
0,revenue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,wholesale,NaN,"11,663.0","12,943.0","12,733.0","13,191.0","10,160.0","11,148.0","12,156.0","11,142.0"
2,retail,NaN,"5,841.0","7,172.0","8,196.0","9,480.0","8,970.0","9,428.0","9,725.0","9,706.0"
3,other businesses,(2),979.0,"1,103.0","986,000",969.0,714.0,658.0,630.0,579.0
4,total net revenue,NaN,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"
5,revenue growth y/y,NaN,NM,14.8%,3.3%,7.9%,-16.1%,7.0%,6.0%,-4.8%
6,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,cost of goods sold,NaN,"(9,500.0)","(10,524.0)","(10,607.0)","(11,347.0)","(9,982.0)","(10,511.0)","(11,863.0)","(11,249.0)"
8,gross profit,NaN,"8,983.0","10,694.0","11,308.0","12,293.0","9,862.0","10,723.0","10,648.0","10,178.0"
9,gross margin %,NaN,48.6%,50.4%,51.6%,52.0%,49.7%,50.5%,47.3%,47.5%


In [587]:
pnl = pnl.drop(columns=["Footnote"])

I removed the footnote column because it was not needed for the analysis and converted the P&L from wide format into tidy format, with one row for each line item and fiscal year.

In [588]:
pnl

,Line Item,FY2016,FY2017,FY2018,FY2019,FY2020,FY2021,FY2022,FY2023
0,revenue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,wholesale,"11,663.0","12,943.0","12,733.0","13,191.0","10,160.0","11,148.0","12,156.0","11,142.0"
2,retail,"5,841.0","7,172.0","8,196.0","9,480.0","8,970.0","9,428.0","9,725.0","9,706.0"
3,other businesses,979.0,"1,103.0","986,000",969.0,714.0,658.0,630.0,579.0
4,total net revenue,"18,483.0","21,218.0","21,915.0","23,640.0","19,844.0","21,234.0","22,511.0","21,427.0"
5,revenue growth y/y,NM,14.8%,3.3%,7.9%,-16.1%,7.0%,6.0%,-4.8%
6,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,cost of goods sold,"(9,500.0)","(10,524.0)","(10,607.0)","(11,347.0)","(9,982.0)","(10,511.0)","(11,863.0)","(11,249.0)"
8,gross profit,"8,983.0","10,694.0","11,308.0","12,293.0","9,862.0","10,723.0","10,648.0","10,178.0"
9,gross margin %,48.6%,50.4%,51.6%,52.0%,49.7%,50.5%,47.3%,47.5%


The original P&L is in wide format, with fiscal years spread across columns.

I converted it into tidy long format using:

In [589]:
tidy_pnl = pnl.melt(
    id_vars=["Line Item"],
    var_name="Fiscal Year",
    value_name="Value"
)


In [590]:
tidy_pnl["Value"] = tidy_pnl["Value"].apply(clean_numeric)


In [591]:
tidy_pnl["Fiscal Year"] = (
    tidy_pnl["Fiscal Year"]
    .str.replace("FY", "", regex=False)
    .astype(int)
)


### Why tidy format is better

Tidy form makes downstream analysis easier because each row represents one line item for one fiscal year, with the line item, fiscal year and value stored in separate columns. This makes filtering and grouping straightforward in Python and allows Excel lookups to use Line Item + Fiscal Year as a reliable key. In Power BI, fiscal year and line item can be treated as dimensions while the value is used as a measure, making charts, filters and calculations easier to build. It also means that new fiscal years can be added without changing the structure of the dataset.

### 6. Correct the unit inconsistency

Total Net Revenue = 23,640
Wholesale         = 13,191
Retail            = 9,480

Other Businesses
= 23,640 - 13,191 - 9,480
= 969


In [592]:
print(tidy_pnl["Fiscal Year"].unique())
print(tidy_pnl["Fiscal Year"].dtype)


[2016 2017 2018 2019 2020 2021 2022 2023]
int64


In [593]:
print(
    tidy_pnl["Line Item"]
    .drop_duplicates()
    .tolist()
)


['revenue', 'wholesale', 'retail', 'other businesses', 'total net revenue', 'revenue growth y/y', 'nan', 'cost of goods sold', 'gross profit', 'gross margin %', 'royalty and commission income', 'other operating income', 'other operating expenses', 'marketing and point-of-sale', 'distribution and selling expense', 'general & administration', 'sundry expenses', 'total other operating exp.', 'goodwill / intangible impairment', 'operating profit (ebit)', 'operating margin %', 'financial income', 'financial expenses', 'income before taxes', 'income tax expense', 'effective tax rate %', 'net income from continuing ops.', 'net income from disc. operations', 'net income (total)', 'less: minority interest', 'net income attrib. to shareholders', 'basic eps', 'diluted eps', 'wtd. avg shares - basic (mn)', 'wtd. avg shares - diluted (mn)', 'dividend per share (dps)']


In [594]:
tidy_pnl[
    tidy_pnl["Line Item"].astype(str).str.contains(
        "Revenue|Wholesale|Retail|Other",
        case=False,
        na=False
    )
].sort_values(["Fiscal Year", "Line Item"])


,Line Item,Fiscal Year,Value
3,other businesses,2016,979.000
12,other operating expenses,2016,NaN
11,other operating income,2016,60.000
2,retail,2016,5841.000
0,revenue,2016,NaN
...,...,...,...
252,revenue,2023,NaN
257,revenue growth y/y,2023,-0.048
256,total net revenue,2023,21427.000
269,total other operating exp.,2023,-9985.000


In [595]:
tidy_pnl["Fiscal Year"] = pd.to_numeric(
    tidy_pnl["Fiscal Year"],
    errors="coerce"
).astype("Int64")


In [596]:
print(tidy_pnl["Fiscal Year"].unique())


<IntegerArray>
[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Length: 8, dtype: Int64


In [597]:
tidy_pnl["Line Item"] = (
    tidy_pnl["Line Item"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)


In [598]:
print(
    tidy_pnl[
        tidy_pnl["Line Item"].isin([
            "Wholesale",
            "Retail",
            "Other Businesses",
            "Total Net Revenue"
        ])
    ].sort_values(["Fiscal Year", "Line Item"])
)


Empty DataFrame
Columns: [Line Item, Fiscal Year, Value]
Index: []


In [599]:
# Get FY2019 values from tidy_pnl
total_net_revenue = tidy_pnl.loc[
    (tidy_pnl["Line Item"] == "total net revenue") &
    (tidy_pnl["Fiscal Year"] == 2019),
    "Value"
].iloc[0]

wholesale = tidy_pnl.loc[
    (tidy_pnl["Line Item"] == "wholesale") &
    (tidy_pnl["Fiscal Year"] == 2019),
    "Value"
].iloc[0]

retail = tidy_pnl.loc[
    (tidy_pnl["Line Item"] == "retail") &
    (tidy_pnl["Fiscal Year"] == 2019),
    "Value"
].iloc[0]

expected_other_businesses = (
    total_net_revenue
    - wholesale
    - retail
)

print("Expected Other Businesses:", expected_other_businesses)

assert np.isclose(expected_other_businesses, 969.0)

print("FY2019 reconciliation passed.")


Expected Other Businesses: 969.0
FY2019 reconciliation passed.


In [600]:
mask = (
    (tidy_pnl["Line Item"] == "Other Businesses") &
    (tidy_pnl["Fiscal Year"] == 2019)
)

tidy_pnl.loc[mask, "Value"] = 969.0


In [601]:
tidy_pnl[
    (tidy_pnl["Line Item"] == "Other Businesses") &
    (tidy_pnl["Fiscal Year"] == 2019)
]


,Line Item,Fiscal Year,Value


In [602]:
tidy_pnl[
    tidy_pnl["Line Item"] == "Other Businesses"
].sort_values("Fiscal Year")


,Line Item,Fiscal Year,Value


In [603]:
mask = (
    (tidy_pnl["Line Item"] == "Other Businesses") &
    (tidy_pnl["Fiscal Year"] == 2018)
)

tidy_pnl.loc[mask, "Value"] = 986.0



Issue:	Unit/value inconsistency

Evidence&Fix:	986,000 conflicts with surrounding years and fails revenue reconciliation	 and it Corrected to 986

The FY2018 Other Businesses value in the raw P&L is shown as:

986,000

However, the supporting REV BY SEGMENT and REV BY GEO sheets both show:

986.0 EUR million

The surrounding years are also around EUR 1,000 million, and the FY2018 Total Net Revenue is EUR 21,915 million.

Therefore, I corrected the FY2018 Other Businesses value from 986,000 to 986.0 EUR million.

This correction is important because leaving 986,000 in the dataset would make the revenue analysis incorrect.

I recorded the correction in the data-quality log rather than silently changing the raw data.

### 7. Reconciliation checks

I performed reconciliation checks after cleaning to make sure that the data remained internally consistent.

**A. Revenue by channel = Total Net Revenue**

Formula:

Wholesale + Retail + Other Businesses = Total Net Revenue

In [604]:
tidy_pnl[
    (tidy_pnl["Fiscal Year"] == 2018) &
    (tidy_pnl["Line Item"].isin([
        "wholesale",
        "retail",
        "other businesses",
        "total net revenue"
    ]))
]


,Line Item,Fiscal Year,Value
73,wholesale,2018,12733.0
74,retail,2018,8196.0
75,other businesses,2018,986000.0
76,total net revenue,2018,21915.0


In [605]:
mask = (
    (tidy_pnl["Line Item"] == "other businesses") &
    (tidy_pnl["Fiscal Year"] == 2018)
)

tidy_pnl.loc[mask, "Value"] = 986


In [606]:
for year in range(2016, 2024):

    wholesale = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "wholesale") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    retail = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "retail") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    other = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "other businesses") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    total_revenue = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "total net revenue") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    calculated = wholesale + retail + other

    assert np.isclose(
        calculated,
        total_revenue,
        atol=0.01
    ), f"Revenue check failed: {year}"

print("Revenue reconciliation passed.")


Revenue reconciliation passed.


**B. Gross Profit check**

Gross Profit = Revenue + COGS

In [607]:
for year in range(2016, 2024):

    revenue = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "total net revenue") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    cogs = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "cost of goods sold") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    gross_profit = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "gross profit") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    calculated = revenue + cogs

    assert np.isclose(
        calculated,
        gross_profit,
        atol=0.01
    ), f"Gross profit check failed: {year}"

print("Gross profit reconciliation passed.")


Gross profit reconciliation passed.


In [608]:
gross_profit

np.float64(10178.0)

**C. EBIT check**
Gross Profit
+ Royalty and Commission Income
+ Other Operating Income
+ Total Other Operating Expenses
+ Goodwill / Intangible Impairment
= EBIT


In [609]:
for year in range(2016, 2024):

    gross_profit = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "gross profit") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    royalty = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "royalty and commission income") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    other_income = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "other operating income") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    operating_expenses = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "total other operating exp.") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    impairment = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "goodwill / intangible impairment") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    # Treat missing/nil impairment as zero
    if pd.isna(impairment):
        impairment = 0

    calculated = (
        gross_profit
        + royalty
        + other_income
        + operating_expenses
        + impairment
    )

    ebit = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "operating profit (ebit)") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    assert np.isclose(
        calculated,
        ebit,
        atol=1
    ), f"EBIT check failed: {year}"

print("EBIT reconciliation passed.")


EBIT reconciliation passed.


ebit

**D. EPS check**

EPS =
Net Income Attributable to Shareholders
÷
Weighted Average Shares


In [610]:
for year in range(2016, 2024):

    net_income = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "net income attrib. to shareholders") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    shares = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "wtd. avg shares - basic (mn)") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    eps = tidy_pnl.loc[
        (tidy_pnl["Line Item"] == "basic eps") &
        (tidy_pnl["Fiscal Year"] == year),
        "Value"
    ].iloc[0]

    calculated_eps = net_income / shares

    assert np.isclose(
        calculated_eps,
        eps,
        atol=0.02
    ), f"EPS check failed: {year}"

print("EPS reconciliation passed.")


EPS reconciliation passed.


In [611]:
eps

np.float64(-0.18)

**8. Export a clean dataset**


In [612]:
tidy_pnl.to_csv(
    "ADS_GY_Equity_Clean_PnL.csv",
    index=False
)


In [613]:
tidy_pnl.to_excel(
    "ADS_GY_Equity_Clean_PnL.xlsx",
    index=False
)


Take the messy Bloomberg-style raw data, clean every problem without destroying information, check that the numbers still make accounting sense, and convert the data into a structured format that Excel and Power BI can use.

After cleaning and validation, I exported the data in a format that can be used by both Excel and Power BI.

In [614]:
print(tidy_pnl.columns.tolist())


['Line Item', 'Fiscal Year', 'Value']


**final clean dataset contain Column :**

Line Item:	Identifies the P&L metric

Fiscal Year:	Identifies the year

Value:	Numeric financial value

with seperate Lookup Table including Line Item and

Statement Group:	Revenue, Cost, Operating Expense, Financial, Subtotal, etc.

Sign Convention:	Identifies whether the item is positive or negative